In [ ]:
import xarray as xr
import os

# ========================
# Settings
# ========================
output_dir = "/Users/szelie/data/unu/terra_climate"
os.makedirs(output_dir, exist_ok=True)

years = range(1985, 2023)
variables = ["tmin", "tmax", "ppt", "pet"]
lat_bounds = (20,5)
lon_bounds = (-95, -65)

# ========================
# Helper function
# ========================
def download_and_crop(var, year):
    url = f"http://thredds.northwestknowledge.net:8080/thredds/dodsC/TERRACLIMATE_ALL/data/TerraClimate_{var}_{year}.nc"
    print(f"🔄 Accessing {url}")
    try:
        ds = xr.open_dataset(url)
        ds = ds.assign_coords(lon=(((ds.lon + 180) % 360) - 180))  # shift 0–360 to -180–180
        ds_crop = ds.sel(lat=slice(*lat_bounds), lon=slice(*lon_bounds))
        out_path = os.path.join(output_dir, f"TerraClimate_{var}_{year}_CA.nc")
        ds_crop.to_netcdf(out_path)
        print(f"✅ Saved to {out_path}")
    except Exception as e:
        print(f"❌ Failed for {var} {year}: {e}")

# ========================
# Run batch download
# ========================
for var in variables:
    for year in years:
        download_and_crop(var, year)


In [ ]:
import xarray as xr
import os
import requests

# ========================
# Settings
# ========================
scenario = "plus2C"
prefix = "2c"
output_dir = "/Users/szelie/data/unu/terra_climate_scenarios_ncss"
os.makedirs(output_dir, exist_ok=True)

years = range(1985, 2015)
variables = ["ppt", "tmin", "tmax", "pet"]
lat_bounds = (20,5)
lon_bounds = (-95, -65)


# ========================
# Helper function
# ========================
def download_and_crop(var, year):
    filename = f"TerraClimate_{prefix}_{var}_{year}.nc"
    url = f"http://thredds.northwestknowledge.net:8080/thredds/fileServer/TERRACLIMATE_ALL/data_{scenario}/{filename}"
    print(f"🔄 Accessing {url}")
    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        with open(filename, "wb") as f:
            f.write(response.content)
        ds = xr.open_dataset(filename)
        ds = ds.assign_coords(lon=(((ds.lon + 180) % 360) - 180))
        ds_crop = ds[var].sel(lat=slice(*lat_bounds), lon=slice(*lon_bounds))
        out_path = os.path.join(output_dir, f"TerraClimate_{scenario}_{var}_{year}_CA.nc")
        ds_crop.to_netcdf(out_path)
        print(f"✅ Saved to {out_path}")
        os.remove(filename)
    except Exception as e:
        print(f"❌ Failed for {var} {year}: {e}")

# ========================
# Run batch download
# ========================
for var in variables:
    for year in years:
        download_and_crop(var, year)
